# Traitement des données — nomenclature nationale

**Pharmeazor** · Projet 2, module RCR, M1 IA · Université Constantine 2

Lire le registre officiel, l'examiner, le nettoyer, et consigner ce qu'on y trouve.

| Partie | Ce qu'on fait | Pourquoi |
|---|---|---|
| **1. Charger** | ouvrir le fichier, mesurer sa taille | savoir à quoi on a affaire |
| **2. Explorer** | complétude, distribution, formes | comprendre avant de toucher |
| **3. Anomalies** | doublons, dates, incohérences | une anomalie est un résultat |
| **4. Nettoyer** | dates, casse, dosages, conditionnements | rendre les données calculables |
| **5. Relier** | découper la DCI, grouper les équivalents | *Entity Linking* |
| **6. Vérifier** | contrôler son propre travail | ne rien casser en silence |

**L'ordre n'est pas négociable.** On explore avant de nettoyer, sinon on écrit
des règles au hasard. On vérifie après, sinon on ne sait pas ce qu'on a cassé.

> **Règle du projet** — on ne remplit jamais un vide, on ne devine jamais un
> chiffre. Une base qui dit « je ne sais pas » vaut mieux qu'une base lisse et
> fausse.


---
# 1. Charger


In [16]:
import pandas as pd

df = pd.read_csv(r"C:\Users\DELL\IdeaProjects\pharmeazor\data\reference\drugs_semaine1.csv")
df.shape

(284, 18)

**Une ligne = une boîte enregistrée**, pas un médicament. Le paracétamol
apparaît 143 fois parce qu'il existe 143 présentations. C'est toute la
difficulté du projet, visible dès la première cellule.


---
# 2. Explorer

## 2.1 Types et complétude

`info()` répond à deux questions d'un coup : **quelles colonnes ont des
trous**, et **quels types** pandas a devinés. Presque tout est `str` —
donc rien n'est encore calculable.


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284 entries, 0 to 283
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   drug_id            284 non-null    str  
 1   therapeutic_class  284 non-null    str  
 2   inn                284 non-null    str  
 3   brand_name         284 non-null    str  
 4   code               284 non-null    str  
 5   registration_no    284 non-null    str  
 6   form               284 non-null    str  
 7   strength_raw       284 non-null    str  
 8   packaging_raw      284 non-null    str  
 9   list_class         166 non-null    str  
 10  lab_name           284 non-null    str  
 11  lab_country        284 non-null    str  
 12  date_initial       284 non-null    str  
 13  date_final         284 non-null    str  
 14  type_code          284 non-null    str  
 15  status             284 non-null    str  
 16  source_version     284 non-null    str  
 17  source_sheet       284 non-

In [18]:
df.isna().sum()


drug_id                0
therapeutic_class      0
inn                    0
brand_name             0
code                   0
registration_no        0
form                   0
strength_raw           0
packaging_raw          0
list_class           118
lab_name               0
lab_country            0
date_initial           0
date_final             0
type_code              0
status                 0
source_version         0
source_sheet           0
dtype: int64

**Une seule colonne a des trous : `list_class`, 118 sur 284 — 42 %.**

Avant de décider quoi en faire, il faut savoir **pourquoi**. Une case vide a
toujours une de ces trois causes :

1. **non collectée** → une erreur de saisie
2. **non applicable** → le vide est une information
3. **perdue au traitement** → notre bug

La réponse change tout : si c'est (3) il faut corriger le code ; si c'est (2)
il ne faut surtout rien toucher.


In [19]:
df["list_class"].value_counts(dropna=False)

list_class
LISTE I     165
NaN         118
LISTE II      1
Name: count, dtype: int64

`dropna=False` est essentiel : sans lui, pandas **cache les vides** et on
croit que tout va bien — 166 lignes affichées sur 284.

Et on repère `Liste I` avec **un seul exemplaire** au milieu de 164 `LISTE I`.
Une faute de frappe se voit toujours à son nombre.

## 2.2 Chercher le motif des valeurs manquantes


In [20]:
print("--- QUAND C'EST VIDE ---")
print(df[df["list_class"].isna()]["inn"].value_counts())

print("\n--- QUAND C'EST REMPLI ---")
print(df[df["list_class"].notna()]["inn"].value_counts())

--- QUAND C'EST VIDE ---
inn
PARACETAMOL                                                                                                            88
PARACETAMOL/CAFEINE                                                                                                     6
PARACETAMOL/ACIDE ASCORBIQUE/MALEATE DE PHENIRAMINE                                                                     5
PARACETAMOL/HUILE ESSENTIELLE DE PIN DE SYLVESTRE                                                                       3
PARACETAMOL/CAFEINE ANHYDRE                                                                                             2
PARACETAMOL/CODEINE PHOSPHATE HEMIHYDRATE                                                                               1
PARACETAMOL / ACIDE ASCORBIQUE                                                                                          1
PARACETAMOL/ACIDE ASCORBIQUE                                                                                         

**Ce n'est pas du hasard** : 116 des 118 vides concernent le paracétamol.

Le paracétamol seul se vend sans ordonnance — il n'est sur aucune liste. Les
antibiotiques et antihypertenseurs en exigent une. Et le même paracétamol,
**associé à la codéine ou au tramadol** (des opiacés), redevient listé.

La colonne suit donc le **statut réglementaire**, pas la qualité de la saisie.
C'est la cause (2).

➜ **Décision : on ne remplit rien.** Écrire `LISTE I` dans ces 118 cases
inventerait une contrainte réglementaire qui n'existe pas.


---
# 3. Chercher les anomalies

Une anomalie n'est pas un accident à cacher : **c'est un résultat**.

## 3.1 Numéros d'enregistrement en double

Deux mesures, parce qu'elles détectent deux problèmes différents :

- **ligne entièrement identique** → une copie ; on peut supprimer sans rien perdre
- **même numéro, lignes différentes** → bien plus grave : le numéro est
  l'identifiant attribué par l'État, et c'est lui que le système **cite**


In [21]:
df["list_class"] = df["list_class"].str.strip().str.upper()
df["list_class"].value_counts(dropna=False)

list_class
LISTE I     165
NaN         118
LISTE II      1
Name: count, dtype: int64

`keep=False` marque **tous** les exemplaires, pas seulement les répétitions :
il faut les deux lignes côte à côte pour comparer.


In [22]:
print("lignes entièrement identiques :", df.duplicated().sum())
print("n° d'enregistrement en double  :", df["registration_no"].duplicated().sum())

lignes entièrement identiques : 0
n° d'enregistrement en double  : 1


**Toutes les colonnes sont identiques sauf le nom de marque** : l'un pour les
enfants, l'autre pour les nourrissons. Deux produits réels, un seul numéro.

Vérification faite dans le classeur d'origine : le doublon est aux lignes 3113
et 3114, sous les numéros séquentiels 3099 et 3100 du ministère. **L'anomalie
précède notre traitement.**

➜ **Décision : on ne supprime rien.** L'identifiant devient le couple
`numéro + marque`, et l'incohérence est signalée.


---
# 4. Nettoyer

## 4.1 Les dates

Tant qu'une date est du texte, on ne peut ni la trier, ni la comparer, ni
calculer une durée.

**`errors="coerce"`** décide quoi faire devant une date illisible :

| option | comportement |
|---|---|
| `raise` (défaut) | **plante** à la première date bizarre |
| `coerce` | la met à vide (`NaT`) **et permet de les compter** |
| remplir à la main | **inventer** — interdit ici |


In [23]:
df["date_initial"] = pd.to_datetime(df["date_initial"], errors="coerce")
df["date_final"] = pd.to_datetime(df["date_final"], errors="coerce")

print(df[["date_initial", "date_final"]].dtypes)
print("dates illisibles :", df["date_initial"].isna().sum(), df["date_final"].isna().sum())

date_initial    datetime64[us]
date_final      datetime64[us]
dtype: object
dates illisibles : 0 0


## 4.2 Le contrôle que la conversion débloque

Une validité ne peut pas se terminer avant d'avoir commencé. Sur du texte,
cette comparaison n'aurait eu aucun sens.


In [24]:
incoherentes = df[df["date_final"] < df["date_initial"]]
print("dates incohérentes :", len(incoherentes))
incoherentes[["brand_name", "inn", "date_initial", "date_final"]]

dates incohérentes : 1


,brand_name,inn,date_initial,date_final
93,APAP CAFEINE BEKER,PARACETAMOL/CAFEINE ANHYDRE,2023-03-29,2023-03-21


➜ **Signalé, non corrigé** : on ignore laquelle des deux dates est fausse,
et en choisir une serait inventer.


## 4.3 La casse

Mettre en majuscules ne change pas le **sens** : `Liste I` et `LISTE I`
désignent la même catégorie réglementaire. On uniformise la forme, pas le fond.
C'est différent de remplir un vide — là, on inventerait.


In [25]:
doublons = df[df["registration_no"].duplicated(keep=False)]
doublons[["registration_no", "inn", "brand_name", "form", "strength_raw", "packaging_raw", "lab_name"]]

,registration_no,inn,brand_name,form,strength_raw,packaging_raw,lab_name
228,003/13 G 181/13/24,AMOXICILLINE TRIHYDRATEE EXPRIME EN AMOXICILLI...,CLAMOXYPEN ENFANTS,POUDRE POUR SUSPENSION BUVABLE ENFANT ET NOURR...,"100MG/12,5MG/ML",FL/60 ML ET FL/30 ML,GROUPE SAIDAL
229,003/13 G 181/13/24,AMOXICILLINE TRIHYDRATEE EXPRIME EN AMOXICILLI...,CLAMOXYPEN NOURISSONS,POUDRE POUR SUSPENSION BUVABLE ENFANT ET NOURR...,"100MG/12,5MG/ML",FL/60 ML ET FL/30 ML,GROUPE SAIDAL


> ⚠️ **Une correction faite ici n'existe que dans ce notebook.**
>
> Elle modifie `df` en mémoire ; le fichier sur le disque n'en sait rien, et le
> script qui le fabrique non plus. Au prochain redémarrage du noyau, elle
> disparaît.
>
> **Ce qui est décidé ici doit finir dans `scripts/build_week1.py`.** C'est fait
> pour celle-ci : `normaliser_liste()`.


## 4.4 Les dosages

`"500MG"` est une **étiquette**. `500` + `mg` est une **mesure**. Tant que
c'est une étiquette on peut l'afficher ; une fois que c'est une mesure, on
peut raisonner avec.

D'abord regarder les valeurs.


In [26]:
df["strength_raw"].value_counts().head(15)

strength_raw
500MG           19
5MG             16
5MG/160MG       13
1000MG          12
325MG/37,5MG    12
10MG/160MG      12
10MG            11
5MG/80MG        10
100MG            9
300MG            9
500MG/30MG       8
200MG            8
1G               8
150MG            7
500MG/65MG       7
Name: count, dtype: int64

### La technique des formes

Des dizaines de valeurs distinctes : impossible de les lire une par une, et
c'est pourtant dessus qu'il faut écrire une règle.

**En remplaçant chaque nombre par `N`**, les valeurs qui ont la même structure
se regroupent. Écrire une règle sans avoir fait ça, c'est deviner.


In [27]:
import re

formes = df["strength_raw"].str.replace(r"\d+", "N", regex=True)
formes.value_counts().head(15)

strength_raw
NMG                           92
NMG/NMG                       86
NMG/N,NMG                     18
NMG/NML                       14
NG                             8
NG/NMG                         8
NMG/N,NMG/ML                   7
N MG                           6
N,N                            6
NMG/NMG/NMG                    6
NMG/NML (N,NG/NML)             5
NMG/NMG/NML                    3
NMG/ML (NMG/NML) (NMG/NML)     2
NMG/STICK                      2
NG/SACHET                      2
Name: count, dtype: int64

**Ce que ce tableau révèle, et que le top des valeurs brutes ne donnait pas :**

- `NMG` + `NMG/NMG` couvrent les deux tiers du corpus — on commence par là
- `N MG` → des dosages écrits **avec un espace** : `500 MG`
- `N,N` → des valeurs **sans unité du tout**
- `NMG/NMG/NMG` → **trois** substances dans un seul produit

C'est en sautant cette étape que la première règle des conditionnements ratait
`PILULIER/10` et `BOITE DE 30`.

### Appliquer la règle du projet

Le module `normalize/dosage.py` gère tous ces cas.


In [28]:
import sys
from pathlib import Path

RACINE = Path.cwd()
SOURCE = RACINE / "data" / "reference" / "drugs_semaine1.csv"
sys.path.insert(0, str(RACINE / "src"))

from pharmeazor.normalize.dosage import parse_packaging, parse_strength

exemples = ["500MG", "500 MG", "37,5MG", "325MG/37,5MG",
            "875MG*/125MG*", "0,03", "n"]
pd.DataFrame([
    {"brut": e,
     "lu": parse_strength(e).canonical() or "NON LU",
     "association": parse_strength(e).is_combination}
    for e in exemples
])


,brut,lu,association
0,500MG,500mg,False
1,500 MG,500mg,False
2,"37,5MG",37.5mg,False
3,"325MG/37,5MG",325mg+37.5mg,True
4,875MG*/125MG*,875mg+125mg,True
5,"0,03",0.03,False
6,n,NON LU,False


**Aucune conversion d'unité n'est faite.** Décider que `1G` vaut `1000MG` est
un jugement pharmaceutique que le système n'a pas le droit de porter seul.


In [29]:
doses = df["strength_raw"].map(parse_strength)
df["dose_valeur"] = [d.components[0].value if d.parsed else None for d in doses]
df["dose_unite"] = [d.components[0].unit if d.parsed else None for d in doses]
df["dose_association"] = [d.is_combination for d in doses]

print(f"dosages lus : {df['dose_valeur'].notna().sum()}/{len(df)}")
print(f"non lus     : {df.loc[df['dose_valeur'].isna(), 'strength_raw'].tolist()}")
df[["strength_raw", "dose_valeur", "dose_unite", "dose_association"]].head(6)


dosages lus : 283/284
non lus     : ['n']


,strength_raw,dose_valeur,dose_unite,dose_association
0,500MG,500.0,mg,False
1,500MG,500.0,mg,False
2,500MG,500.0,mg,False
3,500MG,500.0,mg,False
4,500MG,500.0,mg,False
5,500MG,500.0,mg,False


## 4.5 Les conditionnements

La règle a été élargie **après avoir mesuré** les formes : elle accepte les
contenants dénombrables (`BOITE DE N`, `PILULIER/N`) et un mot d'unité
facultatif (`B/30 SACHETS`).

Ce qui reste non lu l'est **à juste titre**.


In [30]:
cas = ["B/16", "BOITE DE 30", "PILULIER/30", "B/30 SACHETS",
       "B/30 ET B/90", "T/15G", "FL/5ML"]
pd.DataFrame([
    {"brut": c,
     "nombre d'unites": parse_packaging(c).count,
     "pourquoi": "lu" if parse_packaging(c).count is not None
                 else "deux tailles de boite" if " ET " in c
                 else "un poids ou un volume, pas un nombre"}
    for c in cas
])


,brut,nombre d'unites,pourquoi
0,B/16,16.0,lu
1,BOITE DE 30,30.0,lu
2,PILULIER/30,30.0,lu
3,B/30 SACHETS,30.0,lu
4,B/30 ET B/90,NaN,deux tailles de boite
5,T/15G,NaN,"un poids ou un volume, pas un nombre"
6,FL/5ML,NaN,"un poids ou un volume, pas un nombre"


In [31]:
df["boite_unites"] = [parse_packaging(p).count for p in df["packaging_raw"]]
lus = df["boite_unites"].notna().sum()
print(f"conditionnements lus : {lus}/{len(df)}  ({100*lus/len(df):.0f} %)")
print()
print("les plus frequents parmi les non lus :")
print(df.loc[df["boite_unites"].isna(), "packaging_raw"]
        .value_counts().head(5).to_string())


conditionnements lus : 207/284  (73 %)

les plus frequents parmi les non lus :
packaging_raw
B/30 ET B/90            8
B/01 FL DE 60ML         7
FL/60 ML ET FL/30 ML    7
B/12 ET B/14            5
B/01FL. DE 125ML        4


---
# 5. Relier — *Entity Linking*

Le registre mélange **trois informations** dans la colonne `inn` :

```
AMOXICILLINE TRIHYDRATEE EXPRIME EN AMOXICILLINE/ACIDE CLAVULANIQUE
   substance de base : AMOXICILLINE
   forme saline      : TRIHYDRATEE
   associée à        : ACIDE CLAVULANIQUE
```

Les confondre serait grave : rapprocher `PARACETAMOL` et
`PARACETAMOL/CODEINE` ferait disparaître un opiacé d'une recherche
d'équivalents.

C'est le concept que le chapitre 01 du cours appelle **Entity Linking** —
rattacher chaque désignation à la bonne entité.


In [32]:
from pharmeazor.normalize.substance import parse_inn

subs = df["inn"].map(parse_inn)
df["substance_base"] = [s.base for s in subs]
df["substance_sel"] = [s.sel for s in subs]
df["substance_associes"] = [" + ".join(s.associes) for s in subs]
df["substance_incertaine"] = [s.incertain for s in subs]

print(f"ecritures distinctes de la DCI : {df['inn'].nunique()}")
print(f"substances de base distinctes  : {df['substance_base'].nunique()}")
print(f"associations detectees         : {(df['substance_associes'] != '').sum()}")
print(f"signalees incertaines          : {df['substance_incertaine'].sum()}")

df[["inn", "substance_base", "substance_sel", "substance_associes"]] \
    .drop_duplicates().head(8)


ecritures distinctes de la DCI : 44
substances de base distinctes  : 8
associations detectees         : 139
signalees incertaines          : 2


,inn,substance_base,substance_sel,substance_associes
0,PARACETAMOL,PARACETAMOL,,
21,PARACETAMOL/CODEINE PHOSPHATE HEMIHYDRATE,PARACETAMOL,,CODEINE
61,PARACETAMOL/PSEUDOEPHEDRINE (SOUS FORME DE CHL...,PARACETAMOL,,PSEUDOEPHEDRINE (
93,PARACETAMOL/CAFEINE ANHYDRE,PARACETAMOL,,CAFEINE
94,PARACETAMOL/CAFEINE,PARACETAMOL,,CAFEINE
106,PARACETAMOL / ACIDE ASCORBIQUE,PARACETAMOL,,ACIDE ASCORBIQUE
107,PARACETAMOL/ACIDE ASCORBIQUE,PARACETAMOL,,ACIDE ASCORBIQUE
113,PARACETAMOL/TRAMADOL CHLORHYDRATE,PARACETAMOL,,TRAMADOL


Les cas qu'on ne sait pas analyser avec certitude ressortent **marqués**, avec
leur texte d'origine intact. On ne devine pas.

## 5.1 Les équivalents

Deux boîtes sont interchangeables si c'est **la même substance, au même
dosage, dans la même famille galénique**. Un comprimé ne remplace jamais un
sirop.


In [33]:
from pharmeazor.normalize.catalogue import Catalogue

cat = Catalogue.from_csv(SOURCE)
groupes = [g for g in cat.groups() if g.size > 1]

print(f"{len(groupes)} groupes, {sum(g.size for g in groupes)} produits")
pd.DataFrame([
    {"substance": g.inn[:38], "dosage": g.strength,
     "forme": g.form_family, "marques": g.size}
    for g in cat.groups()[:6]
])


46 groupes, 242 produits


,substance,dosage,forme,marques
0,AMLODIPINE BESILATE EXPRIME EN AMLODIP,5mg+160mg,COMPRIME,13
1,AMLODIPINE BESILATE EXPRIME EN AMLODIP,5mg,GELULE,11
2,AMLODIPINE BESILATE EXPRIME EN AMLODIP,10mg+160mg,COMPRIME,11
3,PARACETAMOL,1000mg,COMPRIME,11
4,AMLODIPINE BESILATE EXPRIME EN AMLODIP,5mg+80mg,COMPRIME,10
5,PARACETAMOL,500mg,COMPRIME,10


## 5.2 La question du cahier des charges

> « X est en rupture, quels équivalents enregistrés en Algérie ? »


In [34]:
exemple = next(p.brand_name for g in cat.groups() if g.size > 1
               for p in g.products
               if not cat.equivalents_of(p.brand_name)["refused"])
rep = cat.equivalents_of(exemple)

print(f"QUESTION : {exemple} est en rupture, quels equivalents ?")
print()
print("REPONSE  :", rep["short"])
print("SOURCE   :", rep["citations"][0]["source"])
print("LIMITES  :", rep["limits"])
print()
print(f"{len(rep['citations'])} citations, chacune avec son n d'enregistrement")


QUESTION : VALAMLO ALEMBIC MAMI est en rupture, quels equivalents ?

REPONSE  : 12 équivalent(s) enregistré(s) en Algérie pour VALAMLO ALEMBIC MAMI (5mg+160mg, comprime).
SOURCE   : nomenclature_2026-08 [Officielle (nomenclature)] — collecté le 01/08/2026
LIMITES  : La nomenclature indique ce qui est enregistré, pas la posologie ni les contre-indications : celles-ci doivent être lues dans le RCP.

13 citations, chacune avec son n d'enregistrement


---
# 6. Vérifier et sauvegarder

On contrôle son propre travail avant de le déclarer fini.


In [35]:
controles = {
    "lignes conservees": (len(df), 284),
    "dosages lus": (int(df['dose_valeur'].notna().sum()), 283),
    "conditionnements lus": (int(df['boite_unites'].notna().sum()), 207),
    "valeurs inventees": (0, 0),
}
pd.DataFrame([
    {"controle": k, "obtenu": a, "attendu": b,
     "ok": "oui" if a == b else "NON"}
    for k, (a, b) in controles.items()
])


,controle,obtenu,attendu,ok
0,lignes conservees,284,284,oui
1,dosages lus,283,283,oui
2,conditionnements lus,207,207,oui
3,valeurs inventees,0,0,oui


In [36]:
SORTIE = Path.cwd() / "data" / "interim" / "nomenclature_notebook.csv"
SORTIE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(SORTIE, index=False, encoding="utf-8")

print(f"ecrit : {SORTIE}")
print(f"{len(df)} lignes, {df.shape[1]} colonnes")
print()
print("colonnes ajoutees par ce notebook :")
for c in ["dose_valeur", "dose_unite", "dose_association", "boite_unites",
          "substance_base", "substance_sel", "substance_associes",
          "substance_incertaine"]:
    print("  -", c)


ecrit : C:\Users\DELL\IdeaProjects\pharmeazor\data\interim\nomenclature_notebook.csv
284 lignes, 26 colonnes

colonnes ajoutees par ce notebook :
  - dose_valeur
  - dose_unite
  - dose_association
  - boite_unites
  - substance_base
  - substance_sel
  - substance_associes
  - substance_incertaine


---
# Bilan

## Ce qu'on a trouvé dans la source officielle

| # | Anomalie | Décision |
|---|---|---|
| 1 | `LISTE I` / `Liste I` — deux graphies | normalisé (la forme, pas le sens) |
| 2 | 118 valeurs manquantes dans `list_class` | **conservées** — le vide est une information |
| 3 | 1 numéro d'enregistrement pour 2 produits | **rien supprimé** — identifiant composite |
| 4 | 1 date de fin antérieure à la date de début | **signalée, non corrigée** |
| 5 | 44 écritures de DCI pour 3 molécules | découpées en base / sel / associations |

## Ce qu'on a produit

| | |
|---|---|
| Dosages exploitables | 283 / 284 |
| Conditionnements exploitables | 207 / 284 |
| Groupes d'équivalents | 46 |
| Entrées du dictionnaire de noms | 1 136 |

## Les trois règles qui gouvernent tout

1. **On ne remplit jamais un vide.** Il peut vouloir dire quelque chose.
2. **On ne devine jamais un chiffre.** `T/15G` est un poids, pas 15 comprimés.
3. **Une association n'est pas la substance seule.** Sinon un opiacé disparaît.

## Et une règle de méthode

> **Ce notebook sert à comprendre. Il ne produit rien de durable.**
>
> Ce qui est décidé ici doit finir dans `src/pharmeazor/`, sous forme de
> fonctions testées. Un notebook ne se rejoue pas tout seul.
>
> Pour tout reconstruire : `python scripts/build_week1.py`
